In [ ]:
# Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display Setting
pd.set_option('display.max_columns', None)
%matplotlib inline
print("Libraries loaded successfully")

In [ ]:
# Load Data

df = pd.read_csv('zomato.csv',encoding='latin-1')
country_codes = pd.read_excel('Country-Code.xlsx')

# Check Shape

print("Zomato Dataset Shape:",df.shape)
print("country Codes Shape:",country_codes.shape)

# Preview
df.head()



In [ ]:
# Performing EDA

# Basic Exploration

print("DATASET INFO")
print(df.info())

print("\n MISSING VALUES")
print(df.isnull().sum())

print("\n UNIQUE COUNTRIES")
print(df['Country Code'].value_counts())

In [ ]:
# Merge Country Names
df = df.merge(country_codes, on= 'Country Code', how='left' )

# Filter India Data Only
df_india = df[df['Country Code'] == 1 ].copy()

print("India Restaurant:", len(df_india))
print("\n Top Cities In India:")
print(df_india['City'].value_counts().head(10))

In [ ]:
# Filter Delhi NCR Cities Only

delhi_ncr_cities = ['New Delhi', 'Gurgaon', 'Noida', 'Faridabad', 'Ghaziabad']
df_delhi = df_india[df_india['City'].isin(delhi_ncr_cities)].copy()

print("Delhi NCR Restaurant:",len(df_delhi))
print("\n City Wise Count:")
print(df_delhi['City'].value_counts())

# Drop Columns We Don't Need
df_delhi = df_delhi.drop(columns=['Longitude','Latitude','Switch to order menu',
                                  'Is delivering now','Rating color','Currency'])

print("\n Remaining Columns:", df_delhi.columns.tolist())
print("Final Shape:", df_delhi.shape)

In [ ]:
# Check Aggregate Rating

print("Rating Value Count:")
print(df_delhi['Aggregate rating'].value_counts().sort_index())

# Check cuisines sample
print("\n Sample Cuisines:")
print(df_delhi['Cuisines'].value_counts().head(10))

# Check Duplicates
print("\n Duplicate Rows:",df_delhi.duplicated().sum())

# Rename Columns For Easier Use

df_delhi = df_delhi.rename(columns={'Restaurant ID' : 'restaurant_id',
                                    'Restaurant Name' : 'restaurant_name',
                                    'Country Code' : 'country_code',
                                    'City' : 'city',
                                    'Address' : 'address',
                                    'Locality' : 'locality',
                                    'Locality Verbose' : 'locality_verbose',
                                    'Cuisines' : 'cuisines',
                                    'Average Cost for two' : 'avg_cost_for_two',
                                    'Has Table booking' : 'has_table_booking',
                                    'Has Online delivery' : 'has_online_delivery',
                                    'Price range' : 'price_range',
                                    'Aggregate rating' : 'aggregate_rating',
                                    'Rating text' : 'rating_text',
                                    'Votes' : 'votes',
                                    'Country' : 'country'})

print("\n Cleaned Columns;", df_delhi.columns.tolist())

In [ ]:
# Separate Rated And Unrated Restaurants
df_rated = df_delhi[df_delhi['aggregate_rating']>0].copy()
df_unrated = df_delhi[df_delhi['aggregate_rating']==0].copy()

print("total Restaurants:", len(df_delhi))
print("Rated Restaurants:", len(df_rated))
print("unrated Restaurant:", len(df_unrated))
print("Unrated Percentage:", round(len(df_unrated)/ len(df_delhi)*100,2),"%")


#Extract Primary Cuisine(first cuisine listed)

df_delhi['primary_cuisine'] = df_delhi['cuisines'].str.split(',').str[0].str.strip()
df_rated['primary_cuisine'] = df_rated['cuisines'].str.split(',').str[0].str.strip()

print("\n Top 10 Primary Cuisines In Delhi NCR:")
print(df_delhi['primary_cuisine'].value_counts().head(10))

In [ ]:
# Add Price label column to df_delhi

price_labels = {1:'Budget', 2: 'Affordable', 3:'Premium', 4:'Luxury'}
df_delhi['price_label'] = df_delhi['price_range'].map(price_labels)
df_rated['price_label'] = df_rated['price_range'].map(price_labels)

# Re-export
df_delhi.to_excel('zomato_delhi_cleaned.xlsx',index=False)
df_delhi.to_excel('zomato_delhi_rated.xlsx',index=False)

print("Done - price_label column added")
print(df_delhi['price_label'].value_counts())

### Visualisation

In [ ]:
# Top 10 Cuisines Bar Chart

# visualisation 1 - top 10 cuisines in delhi ncr

plt.Figure(figsize=(12,6))

top_cuisines = df_delhi['primary_cuisine'].value_counts().head(10)

sns.barplot(x=top_cuisines.values,
            y=top_cuisines.index,
            hue=top_cuisines.index,
            palette='Reds_r',
            legend=False)

plt.title('Top 10 Cuisines In Delhi NCR', fontsize=16,fontweight='bold')
plt.xlabel('Number Of Restaurants', fontsize=12)
plt.ylabel('Cuisine Type', fontsize=12)

# Add Value labels on Bars

for i , v in enumerate(top_cuisines.values):
    plt.text(v + 10, i, str(v), va='center',fontsize=10)

plt.tight_layout()
plt.savefig('top_cuisines.png',dpi=150,bbox_inches='tight')
plt.show()

print("Chart Saved Successfully")

In [ ]:
# Online delivery Analysis

# visualisation 2 - Online Delivery vs rating

plt.Figure(figsize=(10,5))

sns.boxplot(x='has_online_delivery',
            y='aggregate_rating',
            data=df_rated,
            hue='has_online_delivery',
            palette={'Yes':'#E23744', 'No':'#2C3E50'},
            legend=False)

plt.title('Impact Of Online Delivery On Restaurant Rating',fontsize=16,fontweight='bold')
plt.xlabel('Has Online Delivery',fontsize=12)
plt.ylabel('Aggregate Rating',fontsize=12)

# Add Mean Values

for i, val in enumerate(['No','Yes']):
    mean_val = df_rated[df_rated['has_online_delivery']==val]['aggregate_rating'].mean()
    plt.text(i,mean_val,f'Mean: {mean_val:.2f}',
             ha='center',fontsize=11,fontweight='bold',color='white')

plt.tight_layout()
plt.savefig('online_delivery_rating.png',dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
# Price Range vs Rating

# Visualisation 3 -Price Range vs Rating

plt.Figure(figsize=(10,5))

price_labels = {1: 'Budget',2: 'Affordable',3: 'Premium',4: 'Luxury'}
df_rated['price_label'] = df_rated['price_range'].map(price_labels)

avg_rating_price = df_rated.groupby('price_label')['aggregate_rating'].mean().reindex(['Budget','Affordable','Premium','Luxury'])

colors = ['#2ecc71','#f39c12','#e74c3c','#8e44ad']

bars = plt.bar(avg_rating_price.index,
               avg_rating_price.values,
               color=colors,
               edgecolor='white',
               linewidth=1.5)

for bar, val in zip(bars, avg_rating_price.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.02,
             f'{val:.2f}',
             ha='center',fontsize=12,
             fontweight='bold')

plt.title('Average Rating by Price Range In Delhi NCR',fontsize=16,fontweight='bold')
plt.xlabel('Price Category',fontsize=12)
plt.ylabel('Average Rating',fontsize=12)
plt.ylim(0,4.5)

plt.tight_layout()
plt.savefig('price_vs_rating.png',dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
# Visualisation 4 - Top 10 Localities In Delhi NCR

plt.figure(figsize=(12,6))

top_localities = df_delhi['locality'].value_counts().head(10)

sns.barplot(x=top_localities.values,
            y=top_localities.index,
            hue=top_localities.index,
            palette='Blues_r',
            legend=False)

plt.title('Top 10 Localities By Restaurant Count In Delhi NCR',
           fontsize=16,fontweight='bold')
plt.xlabel('Number of Restaurants',fontsize=12)
plt.ylabel('Locality',fontsize=12)

for i, v in enumerate(top_localities.values):
    plt.text(v + 5, i, str(v), va='center',fontsize=10)

plt.tight_layout()
plt.savefig('top_localities.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Votes vs Rating

plt.figure(figsize=(10,6))

# filter outliers
df_plot=df_rated[df_rated['votes']<=3000].copy()

plt.scatter(df_rated['votes'],
            df_rated['aggregate_rating'],
            alpha=0.3,
            color='#E23744',
            edgecolors='none',
            s=20)

plt.title('Relationship Between Votes And Rating In Delhi NCR',
           fontsize=16,fontweight='bold')
plt.xlabel('Number of Votes',fontsize=12)
plt.ylabel('Aggregate Rating',fontsize=12)
plt.ylim(1.5,5.5)

# Add Trend line using filtered data
z=np.polyfit(df_rated['votes'],df_rated['aggregate_rating'],1)
p=np.poly1d(z)
x_line= range(0,3001,10)
plt.plot(list(x_line),p(list(x_line)),
         color='#2C3E50',
         linewidth=2.5,
         label=f'Trend Line')

plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig('votes_vs_rating.png',dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
# Export cleaned data for Power BI dashboard

df_delhi.to_excel('zomato_delhi_cleaned.xlsx',index=False)

df_rated.to_excel('zomato_delhi_rated.xlsx',index=False)

print("Files exported successfully")
